**Pipeline stage covered here:** `Goal → Planner → sub-questions`

## 1. Install dependencies

In [ ]:
# !pip install -q groq ddgs streamlit python-dotenv pandas

## 2. Set your Groq API key

Get a free key at [console.groq.com](https://console.groq.com).
Paste it below (or better — use Colab's "Secrets" panel / a `.env` file so you
don't commit it by accident).

In [ ]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## 3. Shared Groq client

Every stage of the pipeline (planner, drafter, critic, reviser) calls this
same helper, so the API key and client setup only live in one place.

In [ ]:
from groq import Groq

_client = None

def get_groq_client() -> Groq:
    """Return a cached Groq client, reading the API key from env."""
    global _client
    if _client is None:
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            raise RuntimeError(
                "GROQ_API_KEY not set. Run the cell above with your key first."
            )
        _client = Groq(api_key=api_key)
    return _client

## 4. Planner

Breaks a research goal into 3-5 focused, independently-searchable sub-questions.
This is the first stage of the pipeline: **Planner → Retriever → Drafter → Critic → Reviser**

In [ ]:
import json

PLANNER_PROMPT = """You are a research planner. Given a research goal, break it down into
3 to 5 specific, non-overlapping sub-questions that, if answered, would fully address the goal.

Rules:
- Each sub-question must be independently searchable (answerable via a web search).
- Avoid overlap between sub-questions.
- Order them from foundational to specific.
- Do not answer the questions yourself.

Return ONLY a JSON array of strings, nothing else. Example:
["What is X?", "How does X compare to Y?", "What are current limitations of X?"]

Research goal: {goal}
"""


def plan(goal: str, model: str = "openai/gpt-oss-120b") -> list[str]:
    """Decompose a research goal into sub-questions.

    Args:
        goal: The user\'s research goal, e.g. "research competitor pricing for X".
        model: Groq model name.

    Returns:
        List of sub-question strings (3-5 items).
    """
    client = get_groq_client()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": PLANNER_PROMPT.format(goal=goal)}],
        temperature=0.5,
    )
    raw = response.choices[0].message.content.strip()

    # Models sometimes wrap JSON in markdown fences -- strip if present.
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw.replace("json", "", 1).strip()

    try:
        sub_questions = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: treat each non-empty line as a sub-question.
        sub_questions = [line.strip("-\u2022 ").strip() for line in raw.splitlines() if line.strip()]

    return sub_questions[:5]

## 5. Testing

Try the planner on a sample goal and check the sub-questions look reasonable
before moving on to Retriever.

In [ ]:
test_goal = "Research competitor pricing for AI note-taking apps"
sub_questions = plan(test_goal)

print(f"Goal: {test_goal}\n")
print("Sub-questions generated:")
for i, q in enumerate(sub_questions, 1):
    print(f"{i}. {q}")